## Mengambil Dataset/Load Dataset

In [117]:
from pathlib import Path
import pandas as pd

files = list(Path("../data/raw").glob("*.csv"))

for f in files:
    print(f.name)

olist_sellers_dataset.csv
product_category_name_translation.csv
olist_orders_dataset.csv
olist_order_items_dataset.csv
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_products_dataset.csv


## Ringkasan Dataset/Dataset Overview

In [118]:
for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * 30)
    print(df.dtypes)


olist_sellers_dataset
------------------------------
seller_id                   str
seller_zip_code_prefix    int64
seller_city                 str
seller_state                str
dtype: object

product_category_name_translation
------------------------------
product_category_name            str
product_category_name_english    str
dtype: object

olist_orders_dataset
------------------------------
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

olist_order_items_dataset
------------------------------
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: objec

## Profiling Data

In [119]:
datasets = {}

for file in Path("../data/raw").glob("*.csv"):
    datasets[file.stem] = pd.read_csv(file)

for name, df in datasets.items():
    print("=" * 40)
    print(name)
    print(df.shape)
    print(df.isnull().sum())

olist_sellers_dataset
(3095, 4)
seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64
product_category_name_translation
(71, 2)
product_category_name            0
product_category_name_english    0
dtype: int64
olist_orders_dataset
(99441, 8)
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
olist_order_items_dataset
(112650, 7)
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64
olist_customers_dataset
(99441, 5)
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city     

In [120]:
datasets = {}

for file in path.Path("../data/raw").glob("*.csv"):
    datasets[file.stem] = pd.read_csv(file)

metadata = {}

for name, df in datasets.items():
    metadata[name] = {
        "rows": len(df),
        "columns": len(df.columns),
        "missing": df.isnull().sum().sum(),
        "duplicates": df.duplicated().sum()
    }



In [121]:
for name, info in metadata.items():
    print(f"datasets : {name}") 
    print(f"missing : {info['missing']}")
    print(f"duplicates : {info['duplicates']}")
    print(f"rows : {info['rows']}")
    print(f"columns : {info['columns']}")
    print("-" * 30)

datasets : olist_sellers_dataset
missing : 0
duplicates : 0
rows : 3095
columns : 4
------------------------------
datasets : product_category_name_translation
missing : 0
duplicates : 0
rows : 71
columns : 2
------------------------------
datasets : olist_orders_dataset
missing : 4908
duplicates : 0
rows : 99441
columns : 8
------------------------------
datasets : olist_order_items_dataset
missing : 0
duplicates : 0
rows : 112650
columns : 7
------------------------------
datasets : olist_customers_dataset
missing : 0
duplicates : 0
rows : 99441
columns : 5
------------------------------
datasets : olist_geolocation_dataset
missing : 0
duplicates : 261831
rows : 1000163
columns : 5
------------------------------
datasets : olist_order_payments_dataset
missing : 0
duplicates : 0
rows : 103886
columns : 5
------------------------------
datasets : olist_order_reviews_dataset
missing : 145903
duplicates : 0
rows : 99224
columns : 7
------------------------------
datasets : olist_products

## Kandidat Primary Key/Candidate Primary Key

In [122]:
datasets["olist_order_items_dataset"]["order_id"].is_unique

False

In [123]:
datasets["olist_sellers_dataset"]["seller_id"].is_unique

True

In [124]:
datasets["olist_customers_dataset"]["customer_unique_id"].is_unique

False

In [125]:
datasets["olist_customers_dataset"]["customer_id"].is_unique

True

In [126]:
datasets["olist_orders_dataset"]["order_id"].is_unique

True

In [127]:
datasets["olist_order_reviews_dataset"]["order_id"].is_unique

False

In [128]:
datasets["olist_order_payments_dataset"]["order_id"].is_unique

False

In [129]:
datasets["olist_geolocation_dataset"]["geolocation_state"].is_unique

False

In [130]:
datasets["product_category_name_translation"]["product_category_name"].is_unique

True

In [131]:
datasets["olist_products_dataset"]["product_id"].is_unique

True

In [132]:
datasets["olist_order_items_dataset"]["order_item_id"].is_unique

False

In [133]:
datasets["olist_order_reviews_dataset"]["review_id"].is_unique

False

## Validasi Foreign Key/Foreign Key Validation

In [134]:
orders = datasets["olist_orders_dataset"]
customers = datasets["olist_customers_dataset"]

orders["customer_id"].isin(customers["customer_id"]).all()

np.True_

In [135]:
items = datasets["olist_order_items_dataset"]
orders = datasets["olist_orders_dataset"]

items["order_id"].isin(orders["order_id"]).all()

np.True_

In [136]:
items = datasets["olist_order_items_dataset"]
seller = datasets["olist_sellers_dataset"]

items["seller_id"].isin(seller["seller_id"]).all()

np.True_

In [137]:
items = datasets["olist_order_items_dataset"]
products = datasets["olist_products_dataset"]

items["product_id"].isin(products["product_id"]).all()

np.True_

In [138]:
products = datasets["olist_products_dataset"]
category = datasets["product_category_name_translation"]

products["product_category_name"].isin(category["product_category_name"]).all()

np.False_

In [139]:
orders = datasets["olist_orders_dataset"]
reviews = datasets["olist_order_reviews_dataset"]

reviews["order_id"].isin(orders["order_id"]).all()

np.True_

## Menemukan anomali/Data Quality Findings

In [140]:
products["product_category_name"].dropna().isin(
    category["product_category_name"]
).all()

np.False_

In [141]:
invalid_category = products.loc[
    ~products["product_category_name"].isin(category["product_category_name"]),
    "product_category_name"
].dropna().unique()

print(invalid_category)

<ArrowStringArray>
['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']
Length: 2, dtype: str


In [142]:
category[
    category["product_category_name"].str.contains("pc", case=False)
]

,product_category_name,product_category_name_english
53,pcs,computers


In [143]:
category[
    category["product_category_name"].str.contains("cozinha", case=False)
]

,product_category_name,product_category_name_english
26,moveis_cozinha_area_de_servico_jantar_e_jardim,kitchen_dining_laundry_garden_furniture


In [144]:
invalid_rows = products.loc[
    ~products["product_category_name"].isin(category["product_category_name"])
]

print(f"Jumlah baris yang gagal: {len(invalid_rows)}")

invalid_rows[
    ["product_id", "product_category_name"]
].head(10)

Jumlah baris yang gagal: 623


,product_id,product_category_name
105,a41e356c76fab66334f36de622ecbd3a,NaN
128,d8dee61c2034d6d075997acef1870e9b,NaN
145,56139431d72cd51f19eb9f7dae4d1617,NaN
154,46b48281eb6d663ced748f324108c733,NaN
197,5fb61f482620cb672f5e586bb132eae9,NaN
244,e10758160da97891c2fdcbc35f0f031d,NaN
294,39e3b9b12cd0bf8ee681bbc1c130feb5,NaN
299,794de06c32a626a5692ff50e4985d36f,NaN
347,7af3e2da474486a3519b0cba9dea8ad9,NaN
428,629beb8e7317703dcc5f35b5463fd20e,NaN


In [145]:
invalid_rows["product_category_name"].value_counts()

product_category_name
portateis_cozinha_e_preparadores_de_alimentos    10
pc_gamer                                          3
Name: count, dtype: int64

In [146]:
products["product_category_name"].isnull().sum()

np.int64(610)

In [147]:
products_not_null = products.dropna(subset=["product_category_name"])

invalid_category = products.loc[
    ~products["product_category_name"].isin(category["product_category_name"]),
    "product_category_name"
].dropna().unique()

print(invalid_category)

<ArrowStringArray>
['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']
Length: 2, dtype: str
